# Fase 2 - Modelado Supervisado: Predicción de la Calidad del Vino Tinto

En este cuaderno aplicaré técnicas de **Aprendizaje Supervisado**. El objetivo es enseñarle a un algoritmo a predecir la calidad del vino basándose en las propiedades químicas que limpiamos en la fase anterior.

Para cumplir con la pauta, implementaré y compararé dos modelos distintos de clasificación, justificando cada paso técnico.

In [1]:
# Clonar repositorio por si se ejecuta en una nueva sesión de Colab
!git clone https://github.com/dongyah/EA2_SCY1101_Calidad_Vino.git
%cd EA2_SCY1101_Calidad_Vino

Cloning into 'EA2_SCY1101_Calidad_Vino'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 86 (delta 23), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 630.37 KiB | 5.48 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/EA2_SCY1101_Calidad_Vino


### 1. Cargar los datos limpios

Lo primero es importar las librerías necesarias y cargar nuestro dataset `winequality_clean.csv`. Como ya nos encargamos de los nulos, duplicados y justificamos los outliers en la Fase 1, esta tabla está lista para ser consumida por el modelo.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Cargar los datos procesados
df = pd.read_csv('data/processed/winequality_clean.csv')
df.head(3)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5


### 2. Separar "X" e "y" (Variables Predictoras y Variable Objetivo)

**Justificación:** Los algoritmos supervisados necesitan saber claramente qué es lo que intentan adivinar y cuáles son las pistas.
* Mi variable **objetivo (y)** es `quality` (calidad del vino), ya que es la nota que el modelo debe aprender a predecir.
* Mis variables **predictoras (X)** son todas las demás columnas químicas (alcohol, azúcar, etc.) que le servirán de pistas.

In [3]:
X = df.drop('quality', axis=1)
y = df['quality']

### 3. División de Entrenamiento y Prueba (Train/Test Split)

**Justificación:** Si le muestro al modelo todos mis vinos de una vez, se los va a aprender de memoria (esto se llama **Sobreajuste u Overfitting**) y cuando le presente un vino nuevo, fallará.
Por eso, divido los datos: como hemos visto en un ejemplo en clase, le entrego el **70% de los datos** para que estudie y aprenda patrones, y me guardo un **30% oculto** para tomarle un examen final y ver qué tan bien generaliza en la realidad.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Datos para entrenar:", len(X_train))
print("Datos para evaluar:", len(X_test))

Datos para entrenar: 951
Datos para evaluar: 408


### 4. Transformación y Escalamiento (El Pipeline)

**Justificación:** En mi dataset hay variables con números muy grandes (como `total sulfur dioxide` que llega casi a 300) y otras con números muy pequeños (como `chlorides` que vale 0.08). Si meto esto directo al algoritmo, el computador creerá erróneamente que el azufre es más importante solo porque el número es mayor.

Para solucionar esto uso **StandardScaler**, que aplasta todas las columnas a una misma escala (media 0 y desviación 1) para que pesen lo mismo. Para automatizar este proceso y evitar fugas de información, lo meteré dentro de un **Pipeline**.

In [5]:
#Dejaré esta celda de código vacía, y construiré las tuberías (pipelines) completas más abajo, justo en el momento de entrenar a cada modelo

### 5. Entrenamiento del Modelo 1: Árbol de Decisión

**Justificación:** El primer modelo que usaré es un **Árbol de Decisión (Decision Tree)**. Lo elegí como modelo base porque es muy visual y crea reglas lógicas simples (ej: si alcohol > 10, entonces calidad 6). Sin embargo, sé por teoría que tiende a sufrir de varianza alta (Overfitting).

In [6]:
# Construir pipeline para el modelo 1
pipe_tree = Pipeline([
    ('escalador', StandardScaler()),
    ('modelo', DecisionTreeClassifier(random_state=42))
])

# Entrenar
pipe_tree.fit(X_train, y_train)

# Predecir y evaluar
pred_tree = pipe_tree.predict(X_test)
print("--- RESULTADOS ÁRBOL DE DECISIÓN ---")
print("Exactitud (Accuracy):", accuracy_score(y_test, pred_tree))
print("\nReporte de métricas:\n", classification_report(y_test, pred_tree, zero_division=0))

--- RESULTADOS ÁRBOL DE DECISIÓN ---
Exactitud (Accuracy): 0.5098039215686274

Reporte de métricas:
               precision    recall  f1-score   support

           3       0.20      0.20      0.20         5
           4       0.12      0.15      0.13        13
           5       0.65      0.60      0.62       172
           6       0.49      0.49      0.49       164
           7       0.39      0.44      0.42        50
           8       0.00      0.00      0.00         4

    accuracy                           0.51       408
   macro avg       0.31      0.31      0.31       408
weighted avg       0.53      0.51      0.52       408



### 6. Entrenamiento del Modelo 2: Random Forest

**Justificación:** Como descubrimos en la Fase 1, este dataset tiene un gran problema de **Desbalanceo de clases** (casi todos los vinos son calidad 5 o 6). El Árbol de Decisión simple sufre mucho con esto.

Para mejorar, implemento un **Random Forest** (Bosque Aleatorio). Al usar múltiples árboles votando juntos en lugar de uno solo, se reduce drásticamente la varianza, se combate el sobreajuste y el modelo maneja mucho mejor las complejas interacciones químicas del vino desbalanceado.

In [7]:
# Construir pipeline para el modelo 2
pipe_forest = Pipeline([
    ('escalador', StandardScaler()),
    ('modelo', RandomForestClassifier(random_state=42))
])

# Entrenar
pipe_forest.fit(X_train, y_train)

# Predecir y evaluar
pred_forest = pipe_forest.predict(X_test)
print("--- RESULTADOS RANDOM FOREST ---")
print("Exactitud (Accuracy):", accuracy_score(y_test, pred_forest))
print("\nReporte de métricas:\n", classification_report(y_test, pred_forest, zero_division=0))

--- RESULTADOS RANDOM FOREST ---
Exactitud (Accuracy): 0.6225490196078431

Reporte de métricas:
               precision    recall  f1-score   support

           3       0.00      0.00      0.00         5
           4       0.00      0.00      0.00        13
           5       0.68      0.72      0.70       172
           6       0.57      0.69      0.63       164
           7       0.64      0.36      0.46        50
           8       0.00      0.00      0.00         4

    accuracy                           0.62       408
   macro avg       0.32      0.29      0.30       408
weighted avg       0.60      0.62      0.60       408



### 7. Comparación y Sincronización en GitHub

**Conclusión:** Al observar las métricas, el modelo Random Forest obtiene una mejor exactitud y un mejor F1-Score en general en comparación con el Árbol de Decisión. Esto demuestra matemáticamente que promediar muchos árboles (Random Forest) es más óptimo para evitar el sobreajuste frente a datos químicos con clases desbalanceadas.

Por lo tanto, declaro ganador al Random Forest y enviaré mi progreso a la nube.

In [8]:
!git add .
!git commit -m "Fase 2: Modelos supervisados, pipelines y evaluacion listos"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@ce696b4adf0e.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
